# Experiment 10: Ensemble Learning Techniques (Random Forest & AdaBoost vs Baseline Logistic Regression)


### 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


### 2. Load and Explore Dataset


In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dataset shape:", X.shape)
print("
Class names (target encoding):", dict(enumerate(data.target_names)))
print("
Target distribution:")
display(y.map({0: "Malignant", 1: "Benign"}).value_counts())

print("Total missing feature values:", int(X.isnull().sum().sum()))


### 3. Split into Training and Testing Sets (80:20 Stratified)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])
print("
Training class counts:")
display(y_train.map({0: "Malignant", 1: "Benign"}).value_counts())
print("Testing class counts:")
display(y_test.map({0: "Malignant", 1: "Benign"}).value_counts())


### 4. Define Baseline and Ensemble Models


In [ ]:
models = {
    "Logistic Regression (Baseline)": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, random_state=42)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        learning_rate=1.0,
        random_state=42
    )
}

print("Models defined:", ", ".join(models.keys()))


### 5. Train Models and Evaluate Predictions


In [ ]:
results = []
trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    trained_models[name] = model
    predictions[name] = y_pred
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (Macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score (Macro)": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results)
print("Performance comparison on held-out test set:")
display(results_df.round(4))


### 6. Compare Test Accuracy


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["Accuracy"], color=["#1f77b4", "#2ca02c", "#ff7f0e"])
plt.ylabel("Test Accuracy")
plt.xlabel("Model")
plt.title("Baseline vs Ensemble Model Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 7. Compare All Macro Metrics


In [ ]:
metric_columns = [
    "Accuracy", "Precision (Macro)", "Recall (Macro)", "F1-score (Macro)"
]

results_df.set_index("Model")[metric_columns].plot(
    kind="bar", figsize=(11, 6)
)
plt.title("Classification Metric Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


### 8. Class-Wise Classification Reports


In [ ]:
for name, y_pred in predictions.items():
    print("
" + "=" * 65)
    print(name)
    print("=" * 65)
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Malignant", "Benign"],
        zero_division=0
    ))


### 9. Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Malignant", "Benign"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)

plt.tight_layout()
plt.show()


### 10. Identify Best Model by Macro F1-Score


In [ ]:
best_index = results_df["F1-score (Macro)"].idxmax()
best_model_name = results_df.loc[best_index, "Model"]
best_f1 = results_df.loc[best_index, "F1-score (Macro)"]

baseline_f1 = results_df.loc[
    results_df["Model"] == "Logistic Regression (Baseline)",
    "F1-score (Macro)"
].iloc[0]

print("Best model by macro F1-score:", best_model_name)
print("Best macro F1-score:", round(best_f1, 4))
print("Baseline macro F1-score:", round(baseline_f1, 4))
print("Difference (best minus baseline):", round(best_f1 - baseline_f1, 4))
